In [1]:
import pandas as pd

reviews = pd.read_csv("data/winemag-data-130k-v2.csv", index_col=0) 
reviews.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


Groupwise analysis

One function we've been using heavily thus far is the value_counts() function. We can replicate what value_counts() does by doing the following:

In [2]:
reviews.groupby("points").points.count()

points
80       397
81       692
82      1836
83      3025
84      6480
85      9530
86     12600
87     16933
88     17207
89     12226
90     15410
91     11359
92      9613
93      6489
94      3758
95      1535
96       523
97       229
98        77
99        33
100       19
Name: points, dtype: int64

groupby() created a group of reviews which allotted the same point values to the given wines. Then, for each of these groups, we grabbed the points() column and counted how many times it appeared. value_counts() is just a shortcut to this groupby() operation.

We can use any of the summary functions we've used before with this data. For example, to get the cheapest wine in each point value category, we can do the following:

In [3]:
reviews.groupby("points").price.min()

points
80      5.0
81      5.0
82      4.0
83      4.0
84      4.0
85      4.0
86      4.0
87      5.0
88      6.0
89      7.0
90      8.0
91      7.0
92     11.0
93     12.0
94     13.0
95     20.0
96     20.0
97     35.0
98     50.0
99     44.0
100    80.0
Name: price, dtype: float64

You can think of each group we generate as being a slice of our DataFrame containing only data with values that match. This DataFrame is accessible to us directly using the apply() method, and we can then manipulate the data in any way we see fit. For example, here's one way of selecting the name of the first wine reviewed from each winery in the dataset:

In [4]:
reviews.groupby('winery').apply(lambda df: df.title.iloc[0])

winery
1+1=3                                     1+1=3 NV Rosé Sparkling (Cava)
10 Knots                            10 Knots 2010 Viognier (Paso Robles)
100 Percent Wine              100 Percent Wine 2015 Moscato (California)
1000 Stories           1000 Stories 2013 Bourbon Barrel Aged Zinfande...
1070 Green                  1070 Green 2011 Sauvignon Blanc (Rutherford)
                                             ...                        
Órale                       Órale 2011 Cabronita Red (Santa Ynez Valley)
Öko                    Öko 2013 Made With Organically Grown Grapes Ma...
Ökonomierat Rebholz    Ökonomierat Rebholz 2007 Von Rotliegenden Spät...
àMaurice               àMaurice 2013 Fred Estate Syrah (Walla Walla V...
Štoka                                    Štoka 2009 Izbrani Teran (Kras)
Length: 16757, dtype: str

## Understanding `iloc` in Pandas

A common confusion when learning pandas is how `iloc` indexing works.

### 1. `iloc` does *not* always need two inputs

`iloc` indexes **by integer position**. The general form is:

```python
df.iloc[row_indexer, column_indexer]
```

However, the **column part is optional**.

- If you provide **one argument**, it selects **rows only**.
- If you provide **two arguments**, it selects **rows and columns**.

Examples:

```python
df.iloc[0]        # first row (returns a Series)
df.iloc[0:5]      # first 5 rows
df.iloc[0, 1]     # row 0, column 1
df.iloc[:, 1]     # all rows, column 1
```

---

### 2. Understanding the Kaggle line

```python
reviews.groupby('winery').apply(lambda df: df.title.iloc[0])
```

Breakdown:

1. `reviews.groupby('winery')`  
   Splits the dataframe into **groups**, one per winery.

2. `.apply(lambda df: ...)`  
   For each group, pandas passes a **sub-DataFrame** (`df`) to the lambda function.

3. `df.title`  
   Selects the **title column**, returning a **Series**.

4. `df.title.iloc[0]`  
   Selects the **first row of that Series**.

Since `df.title` is already a **Series**, there is only **one axis (rows)**. Therefore:

```python
.iloc[0]
```

means **the first element of the Series**.

---

### 3. Equivalent alternatives

These would produce the same result:

```python
lambda df: df.title.iloc[0]
lambda df: df.title.values[0]
lambda df: df.iloc[0]['title']
```

The Kaggle version is usually the **cleanest and most readable**.

---

### Summary

- `iloc` can take **one or two arguments**.
- One argument → selects **rows only**.
- In this example, `df.title` is a **Series**, so `.iloc[0]` selects the **first value** in that Series.

For even more fine-grained control, you can also group by more than one column. For an example, here's how we would pick out the best wine by country and province:

In [5]:
reviews.groupby(['country', 'province']).apply(lambda df: df.loc[df.points.idxmax()])

description  \
country   province                                                              
Argentina Mendoza Province  If the color doesn't tell the full story, the ...   
          Other             Take note, this could be the best wine Colomé ...   
Armenia   Armenia           Deep salmon in color, this wine offers a bouqu...   
Australia Australia Other   Writes the book on how to make a wine filled w...   
          New South Wales   De Bortoli's Noble One is as good as ever in 2...   
...                                                                       ...   
Uruguay   Juanico           This mature Bordeaux-style blend is earthy on ...   
          Montevideo        A rich, heady bouquet offers aromas of blackbe...   
          Progreso          Rusty in color but deep and complex in nature,...   
          San Jose          Baked, sweet, heavy aromas turn earthy with ti...   
          Uruguay           Cherry and berry aromas are ripe, healthy and ...   

                                                        designation  points  \
country   province                                                            
Argentina Mendoza Province                         Nicasia Vineyard      97   
          Other                                             Reserva      95   
Armenia   Armenia                                    Estate Bottled      88   
Australia Australia Other                             Sarah's Blend      93   
          New South Wales                        Noble One Bortytis      94   
...                                                             ...     ...   
Uruguay   Juanico                  Preludio Barrel Select Lote N 77      90   
          Montevideo        Monte Vide Eu Tannat-Merlot-Tempranillo      91   
          Progreso                   Etxe Oneko Fortified Sweet Red      90   
          San Jose                         El Preciado Gran Reserva      87   
          Uruguay                         Blend 002 Limited Edition      91   

                            price                 region_1 region_2  \
country   province                                                    
Argentina Mendoza Province  120.0                  Mendoza      NaN   
          Other              90.0                    Salta      NaN   
Armenia   Armenia            15.0                      NaN      NaN   
Australia Australia Other    15.0  South Eastern Australia      NaN   
          New South Wales    32.0          New South Wales      NaN   
...                           ...                      ...      ...   
Uruguay   Juanico            45.0                      NaN      NaN   
          Montevideo         60.0                      NaN      NaN   
          Progreso           46.0                      NaN      NaN   
          San Jose           50.0                      NaN      NaN   
          Uruguay            22.0                      NaN      NaN   

                                  taster_name taster_twitter_handle  \
country   province                                                    
Argentina Mendoza Province  Michael Schachner           @wineschach   
          Other             Michael Schachner           @wineschach   
Armenia   Armenia               Mike DeSimone        @worldwineguys   
Australia Australia Other                 NaN                   NaN   
          New South Wales      Joe Czerwinski                @JoeCz   
...                                       ...                   ...   
Uruguay   Juanico           Michael Schachner           @wineschach   
          Montevideo        Michael Schachner           @wineschach   
          Progreso          Michael Schachner           @wineschach   
          San Jose          Michael Schachner           @wineschach   
          Uruguay           Michael Schachner           @wineschach   

                                                                        title  \
country   province                                     

## Why doesn't this return a Series?

```python
reviews.groupby(['country', 'province']).apply(
    lambda df: df.loc[df.points.idxmax()]
)
```

At first glance, it seems like this should return a **Series**, because:

```python
df.loc[row_label]
```

normally returns a **Series** when selecting a single row from a DataFrame.

The key to understanding this behavior is how **`groupby().apply()` combines results**.

---

## 1. What `df.loc[df.points.idxmax()]` returns

Inside the lambda function:

```python
df.loc[df.points.idxmax()]
```

- `df.points.idxmax()`  
  Returns the **index label of the row with the maximum `points`** in that group.

- `df.loc[...]`  
  Selects that **single row** from the DataFrame.

Selecting a single row from a DataFrame returns a **Series**:

```python
type(df.loc[df.points.idxmax()])  # pandas.Series
```

So the lambda function **does return a Series**.

---

## 2. What `groupby().apply()` does with those Series

Your full code:

```python
reviews.groupby(['country', 'province']).apply(
    lambda df: df.loc[df.points.idxmax()]
)
```

For **each group**, the lambda returns a **Series** representing the row with the highest `points`.

Conceptually, this looks like:

```
Group A → Series
Group B → Series
Group C → Series
```

When `apply()` receives **multiple Series with the same index (same columns)**, pandas **automatically combines them into a DataFrame**.

In other words, pandas stacks them like rows:

```
Series 1
Series 2
Series 3
↓
DataFrame
```

So the final output becomes a **DataFrame**, not a Series.

---

## 3. Example of what pandas constructs

Imagine each group returns something like:

```
title      Wine A
points        95
price         40
```

Pandas combines them into:

```
                     title   points  price
country province
US      California  Wine A     95      40
France  Bordeaux    Wine B     97      50
Italy   Tuscany     Wine C     96      45
```

The result is a **DataFrame with a MultiIndex** (`country`, `province`).

---

## 4. Mental model for `groupby().apply()`

You can think of `apply()` like this:

```
group → function → result
group → function → result
group → function → result
```

Then pandas **stitches the results together** depending on what the function returns:

- **Scalar values** → combined into a **Series**
- **Series** → combined into a **DataFrame**
- **DataFrames** → combined into a **larger DataFrame**

---

## Summary

- `df.loc[row_label]` returns a **Series** when selecting a single row.
- Your lambda returns **one Series per group**.
- `groupby().apply()` **combines those Series into a DataFrame**.
```

### Understanding `groupby().apply()` in this example

```python
reviews.groupby('winery').apply(lambda df: df.title.iloc[0])
```

#### 1. `groupby('winery')`
- Splits the DataFrame into groups based on the `winery` column.
- Each group passed to the lambda function is a **DataFrame** (`df`).

#### 2. Inside the lambda function
```python
df.title
```
- This selects the `title` column from the group.
- The result is a **Series**.

#### 3. `.iloc[0]`
```python
df.title.iloc[0]
```
- `.iloc[0]` selects the **first value** from that Series.
- This returns a **scalar value** (a single string), not a Series.

#### 4. What `apply()` collects
- The lambda function returns **one scalar per group**.
- When `apply()` collects scalar outputs from each group, it combines them into a **Series**.

Example result:

```python
winery
WineryA    "Wine Title 1"
WineryB    "Wine Title 2"
WineryC    "Wine Title 3"
dtype: object
```

---

### Why the result is not a DataFrame

`groupby().apply()` **does not force the output type**.  
Instead, the final structure depends on what the function returns for each group.

| Return type from function | Result of `apply()` |
|---------------------------|---------------------|
| Scalar                    | Series              |
| Series                    | DataFrame           |
| DataFrame                 | Concatenated DataFrame |

In this case, the function returns a **scalar**, so `apply()` returns a **Series**.

## Behavior of `df.loc[]` with Single vs Multiple Rows

The type returned by `df.loc[]` depends on **how many rows are selected**.

### 1. Selecting a single row → returns a Series

If the row selector resolves to **one row**, pandas returns a **Series**.

Example:

```python
df.loc[5]
```

Output type:

```python
pandas.Series
```

This happens because pandas interprets the result as **one record**, which naturally maps to a Series.

Example structure:

```
title      Wine A
points        95
price         40
Name: 5
```

---

### 2. Selecting multiple rows → returns a DataFrame

If the row selector resolves to **multiple rows**, pandas returns a **DataFrame**.

Example:

```python
df.loc[[5, 8]]
```

Output type:

```python
pandas.DataFrame
```

Example structure:

```
    title     points  price
5   Wine A       95     40
8   Wine B       92     35
```

---

### 3. Range selection also returns a DataFrame

```python
df.loc[5:8]
```

Output:

```
    title     points  price
5   Wine A       95     40
6   Wine C       91     32
7   Wine D       90     30
8   Wine B       92     35
```

---

### 4. Boolean filtering returns a DataFrame

```python
df.loc[df.points > 95]
```

Since this may match **multiple rows**, the result is a **DataFrame**.

---

## Summary Rule

| Selection Type | Result |
|---|---|
| One row | `Series` |
| Multiple rows | `DataFrame` |

So in your earlier example:

```python
df.loc[df.points.idxmax()]
```

- `idxmax()` returns **one index**
- Therefore `loc[]` selects **one row**
- Result → **Series**

Another groupby() method worth mentioning is agg(), which lets you run a bunch of different functions on your DataFrame simultaneously. For example, we can generate a simple statistical summary of the dataset as follows:

In [6]:
reviews.groupby(['country']).price.agg([len, min, max])

,len,min,max
country,,,
Argentina,3800,4.0,230.0
Armenia,2,14.0,15.0
Australia,2329,5.0,850.0
Austria,3345,7.0,1100.0
Bosnia and Herzegovina,2,12.0,13.0
Brazil,52,10.0,60.0
Bulgaria,141,8.0,100.0
Canada,257,12.0,120.0
Chile,4472,5.0,400.0


Sorting

Looking again at countries_reviewed we can see that grouping returns data in index order, not in value order. That is to say, when outputting the result of a groupby, the order of the rows is dependent on the values in the index, not in the data.

To get data in the order want it in we can sort it ourselves. The sort_values() method is handy for this.

In [7]:
countries_reviewed = reviews.groupby(['country', 'province']).description.agg([len])
countries_reviewed

len
country   province              
Argentina Mendoza Province  3264
          Other              536
Armenia   Armenia              2
Australia Australia Other    245
          New South Wales     85
...                          ...
Uruguay   Juanico             12
          Montevideo          11
          Progreso            11
          San Jose             3
          Uruguay             24

[425 rows x 1 columns]

In [8]:
countries_reviewed = countries_reviewed.reset_index()
countries_reviewed

,country,province,len
0,Argentina,Mendoza Province,3264
1,Argentina,Other,536
2,Armenia,Armenia,2
3,Australia,Australia Other,245
4,Australia,New South Wales,85
...,...,...,...
420,Uruguay,Juanico,12
421,Uruguay,Montevideo,11
422,Uruguay,Progreso,11
423,Uruguay,San Jose,3


In [9]:
countries_reviewed.sort_values(by="len")

,country,province,len
386,Turkey,Elazığ-Diyarbakir,1
389,Turkey,Urla-Thrace,1
395,US,Hawaii,1
357,South Africa,Piekenierskloof,1
354,South Africa,Paardeberg,1
...,...,...,...
409,US,Oregon,5373
227,Italy,Tuscany,5897
118,France,Bordeaux,5941
415,US,Washington,8639


sort_values() defaults to an ascending sort, where the lowest values go first. However, most of the time we want a descending sort, where the higher numbers go first. That goes thusly:

In [10]:
countries_reviewed.sort_values(by="len", ascending=False)

,country,province,len
392,US,California,36247
415,US,Washington,8639
118,France,Bordeaux,5941
227,Italy,Tuscany,5897
409,US,Oregon,5373
...,...,...,...
389,Turkey,Urla-Thrace,1
48,Canada,Canada Other,1
40,Brazil,Serra do Sudeste,1
395,US,Hawaii,1


To sort by index values, use the companion method sort_index(). This method has the same arguments and default order:



In [12]:
countries_reviewed.sort_index()

,country,province,len
0,Argentina,Mendoza Province,3264
1,Argentina,Other,536
2,Armenia,Armenia,2
3,Australia,Australia Other,245
4,Australia,New South Wales,85
...,...,...,...
420,Uruguay,Juanico,12
421,Uruguay,Montevideo,11
422,Uruguay,Progreso,11
423,Uruguay,San Jose,3


Finally, know that you can sort by more than one column at a time:

In [13]:
countries_reviewed.sort_values(by=["country", "len"])

,country,province,len
1,Argentina,Other,536
0,Argentina,Mendoza Province,3264
2,Armenia,Armenia,2
6,Australia,Tasmania,42
4,Australia,New South Wales,85
...,...,...,...
421,Uruguay,Montevideo,11
422,Uruguay,Progreso,11
420,Uruguay,Juanico,12
424,Uruguay,Uruguay,24


# Exercises

## 1.
Who are the most common wine reviewers in the dataset? Create a `Series` whose index is the `taster_twitter_handle` category from the dataset, and whose values count how many reviews each person wrote.

In [14]:
reviews.groupby("taster_twitter_handle")["taster_twitter_handle"].count()

taster_twitter_handle
@AnneInVino          3685
@JoeCz               5147
@bkfiona               27
@gordone_cellars     4177
@kerinokeefe        10776
@laurbuzz            1835
@mattkettmann        6332
@paulgwine           9532
@suskostrzewa        1085
@vboone              9537
@vossroger          25514
@wawinereport        4966
@wineschach         15134
@winewchristina         6
@worldwineguys       1005
Name: taster_twitter_handle, dtype: int64

## 2.
What is the best wine I can buy for a given amount of money? Create a `Series` whose index is wine prices and whose values is the maximum number of points a wine costing that much was given in a review. Sort the values by price, ascending (so that `4.0` dollars is at the top and `3300.0` dollars is at the bottom).

Hint: Use max() and sort_index(). The relevant columns in the DataFrame are price and points.

In [24]:
reviews.groupby('price')["points"].max().sort_index()

price
4.0       86
5.0       87
6.0       88
7.0       91
8.0       91
          ..
1900.0    98
2000.0    97
2013.0    91
2500.0    96
3300.0    88
Name: points, Length: 390, dtype: int64

## 3.
What are the minimum and maximum prices for each `variety` of wine? Create a `DataFrame` whose index is the `variety` category from the dataset and whose values are the `min` and `max` values thereof.

In [ ]:
reviews.groupby("variety")["price"].agg([min, max])

,max,min
variety,,
Abouriou,75.0,15.0
Agiorgitiko,66.0,10.0
Aglianico,180.0,6.0
Aidani,27.0,27.0
Airen,10.0,8.0
...,...,...
Zinfandel,100.0,5.0
Zlahtina,16.0,13.0
Zweigelt,70.0,9.0


## 4.
What are the most expensive wine varieties? Create a variable `sorted_varieties` containing a copy of the dataframe from the previous question where varieties are sorted in descending order based on minimum price, then on maximum price (to break ties).

Hint: Use sort_values(), and provide a list of names to sort by.

In [ ]:
sorted_varieties = reviews.groupby("variety")["price"].agg([min, max])
sorted_varieties = sorted_varieties.sort_values(["min", "max"], ascending=False)
sorted_varieties

,max,min
variety,,
Ramisco,495.0,495.0
Terrantez,236.0,236.0
Francisa,160.0,160.0
Rosenmuskateller,150.0,150.0
Tinta Negra Mole,112.0,112.0
...,...,...
Vital,NaN,NaN
White Blend,NaN,NaN
White Port,NaN,NaN


## 5.
Create a `Series` whose index is reviewers and whose values is the average review score given out by that reviewer. Hint: you will need the `taster_name` and `points` columns.

In [29]:
reviews.groupby("taster_name")["points"].mean()

taster_name
Alexander Peartree    85.855422
Anna Lee C. Iijima    88.415629
Anne Krebiehl MW      90.562551
Carrie Dykes          86.395683
Christina Pickard     87.833333
Fiona Adams           86.888889
Jeff Jenssen          88.319756
Jim Gordon            88.626287
Joe Czerwinski        88.536235
Kerin O’Keefe         88.867947
Lauren Buzzeo         87.739510
Matt Kettmann         90.008686
Michael Schachner     86.907493
Mike DeSimone         89.101167
Paul Gregutt          89.082564
Roger Voss            88.708003
Sean P. Sullivan      88.755739
Susan Kostrzewa       86.609217
Virginie Boone        89.213379
Name: points, dtype: float64

## 6.
What combination of countries and varieties are most common? Create a `Series` whose index is a `MultiIndex`of `{country, variety}` pairs. For example, a pinot noir produced in the US should map to `{"US", "Pinot Noir"}`. Sort the values in the `Series` in descending order based on wine count.

In [32]:
reviews.groupby(["country", "variety"]).winery.count().sort_values()

country  variety                 
France   Prunelard                      1
         Petite Verdot                  1
         Petit Verdot                   1
         Petit Meslier                  1
         Petit Courbu                   1
                                     ... 
Italy    Red Blend                   3624
France   Bordeaux-style Red Blend    4725
US       Chardonnay                  6801
         Cabernet Sauvignon          7315
         Pinot Noir                  9885
Name: winery, Length: 1612, dtype: int64

My experience is that, after groupby() on desired column(s), most of the times, we have to select a column after that in order to perform the requried operation (count, mean, agg these all require a column, you cannot perform these operations on a dataframe which groupby() returns). 